# Modul 8: Analiza sceny z Gemini Robotics-ER API

W tym notebooku uzyjemy **Gemini Robotics-ER 1.5** (publiczne API Google) do analizy obrazu z kamery robota humanoidalnego.

## Czym jest Gemini Robotics-ER?

Google oferuje kilka modeli Gemini zwiazanych z robotyka:

| Model | Dostepnosc | Opis |
|-------|-----------|------|
| **Gemini Robotics (VLA)** | NIE publiczny | Pelny model VLA — generuje akcje motoryczne |
| **Gemini Robotics On-Device** | Waitlist | Wersja on-device do deployu na robocie |
| **Gemini Robotics-ER 1.5** | **PUBLICZNY** | Rozumienie sceny i planowanie — to uzywamy! |
| **Gemini 3.1 Flash-Lite** | Publiczny | Lekki model, $0.25/1M tokenow |

**Wazne:** Gemini Robotics-ER to **NIE jest pelny VLA**. Nie generuje bezposrednio akcji motorycznych (pozycji stawow). Sluzy jako warstwa percepcji i planowania — rozumie scene i generuje plan wysokopoziomowy, ktory nastepnie realizuje wlasciwy VLA (np. GR00T N1.6) i kontroler niskiego poziomu (WBC).

In [ ]:
!pip install -q google-generativeai matplotlib Pillow

## Konfiguracja API

Potrzebujesz klucza Google API z dostepem do Gemini. Mozesz go uzyskac bezplatnie:

1. Wejdz na [Google AI Studio](https://aistudio.google.com/apikey)
2. Utworz nowy klucz API
3. Dodaj go w Colab: **Secrets** (ikona klucza w lewym panelu) → dodaj `GOOGLE_API_KEY`

Alternatywnie — zostaniesz poproszony o wpisanie klucza recznie.

In [ ]:
import os
from google.colab import userdata

try:
    api_key = userdata.get('GOOGLE_API_KEY')
    print("Klucz API zaladowany z Colab Secrets.")
except Exception:
    import getpass
    api_key = getpass.getpass("Podaj Google API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key
print("API Key ustawiony.")

## Analiza sceny — obraz z kamery robota

Pobieramy przykladowy obraz przedstawiajacy scene z perspektywy kamery robota. W rzeczywistym scenariuszu bylby to obraz z kamery zamontowanej na glowie robota G1.

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
from PIL import Image

# Pobierz przykladowy obraz sceny robota (stol z obiektami)
image_url = "https://images.unsplash.com/photo-1606567595334-d39972c85dbe?w=640"
image_path = "robot_view.jpg"

try:
    urllib.request.urlretrieve(image_url, image_path)
    print(f"Pobrano obraz: {image_path}")
except Exception as e:
    print(f"Nie udalo sie pobrac obrazu: {e}")
    print("Mozesz recznie wgrac obraz jako 'robot_view.jpg'")

# Wyswietl obraz
img = Image.open(image_path)
plt.figure(figsize=(10, 7))
plt.imshow(img)
plt.title("Widok z kamery robota")
plt.axis("off")
plt.show()
print(f"Rozmiar obrazu: {img.size}")

In [ ]:
import google.generativeai as genai
import PIL.Image

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.0-flash")

img = PIL.Image.open("robot_view.jpg")

prompt = """Jestes systemem percepcji robota humanoidalnego G1.
Analizujesz obraz z kamery zamontowanej na glowie robota.

Opisz scene i wygeneruj plan pick-and-place w formacie JSON:
{
  "scene_description": "krotki opis sceny",
  "detected_objects": [
    {"name": "obiekt", "position": "opis pozycji", "graspable": true}
  ],
  "plan": [
    {"step": 1, "action": "navigate_to", "target": "stol"},
    {"step": 2, "action": "grasp", "target": "kubek"},
    {"step": 3, "action": "navigate_to", "target": "cel"},
    {"step": 4, "action": "place", "target": "kubek"}
  ]
}

Odpowiedz TYLKO w formacie JSON, bez dodatkowego tekstu."""

print("Wysylam obraz do Gemini...")
response = model.generate_content([img, prompt])
print("\n=== Odpowiedz Gemini ===")
print(response.text)

## Parsowanie planu

Wyciagamy strukturalny plan z odpowiedzi JSON i wyswietlamy go w czytelnej formie.

In [ ]:
import json
import re

# Parsuj JSON z odpowiedzi (obsluz markdown code blocks)
raw_text = response.text.strip()
# Usun ewentualne oznaczenia ```json ... ```
json_match = re.search(r'```(?:json)?\s*(.+?)\s*```', raw_text, re.DOTALL)
if json_match:
    json_text = json_match.group(1)
else:
    json_text = raw_text

try:
    analysis = json.loads(json_text)
except json.JSONDecodeError as e:
    print(f"Blad parsowania JSON: {e}")
    print(f"Surowy tekst:\n{raw_text}")
    analysis = {}

if analysis:
    # Opis sceny
    print("=" * 60)
    print(f"OPIS SCENY: {analysis.get('scene_description', 'brak')}")
    print("=" * 60)

    # Wykryte obiekty
    print(f"\n{'Obiekt':<20} {'Pozycja':<25} {'Chwytny'}")
    print("-" * 55)
    for obj in analysis.get("detected_objects", []):
        graspable = "TAK" if obj.get("graspable") else "NIE"
        print(f"{obj['name']:<20} {obj.get('position', '?'):<25} {graspable}")

    # Plan
    print(f"\n{'Krok':<6} {'Akcja':<20} {'Cel'}")
    print("-" * 45)
    for step in analysis.get("plan", []):
        print(f"{step['step']:<6} {step['action']:<20} {step['target']}")

    print(f"\nLaczna liczba krokow: {len(analysis.get('plan', []))}")

## Trojwarstwowa architektura

W realnym systemie robotycznym Gemini-ER pelni role **warstwy planowania** w trojwarstwowej architekturze:

```
Warstwa 3 — Planowanie (1-5 Hz)
┌─────────────────────────────────────────┐
│  Gemini Robotics-ER 1.5                 │
│  - Analiza sceny (obraz → opis)         │
│  - Plan pick-and-place (JSON)           │
│  - Re-planning gdy cos sie zmieni       │
└──────────────┬──────────────────────────┘
               │ "podejdz do stolu, chwyc kubek"
               ▼
Warstwa 2 — Akcje motoryczne (10 Hz)
┌─────────────────────────────────────────┐
│  GR00T N1.6 / UnifoLM-VLA-0            │
│  - Zamienia plan na pozycje stawow      │
│  - Action chunking (8-16 krokow)       │
│  - Uzywa kamery + propriocepcji        │
└──────────────┬──────────────────────────┘
               │ joint positions [14 DOF]
               ▼
Warstwa 1 — Wykonanie (200-500 Hz)
┌─────────────────────────────────────────┐
│  WBC / Gait Policy / IK                │
│  - Kontrola momentow na silnikach       │
│  - Utrzymanie rownowagi                 │
│  - Fizyczna realizacja ruchu            │
└─────────────────────────────────────────┘
```

**Kazda warstwa dziala z inna czestotliwoscia** — Gemini planuje raz na kilka sekund, VLA generuje akcje 10 razy na sekunde, a kontroler niskiego poziomu dziala setki razy na sekunde.

## Porownanie modeli Gemini

Google udostepnia rozne warianty Gemini — nie wszystkie sa publiczne i nie wszystkie nadaja sie do robotyki.

In [ ]:
import pandas as pd

data = {
    "Model": [
        "Gemini Robotics (VLA)",
        "Gemini Robotics On-Device",
        "Gemini Robotics-ER 1.5",
        "Gemini 2.0 Flash",
        "Gemini 3.1 Flash-Lite",
    ],
    "Dostepnosc": [
        "NIE publiczny",
        "Waitlist",
        "PUBLICZNY",
        "PUBLICZNY",
        "PUBLICZNY",
    ],
    "Koszt (1M tokenow)": [
        "N/A",
        "N/A",
        "Darmowy tier",
        "Darmowy tier",
        "$0.25",
    ],
    "Generuje akcje?": [
        "TAK (pelny VLA)",
        "TAK (on-device)",
        "NIE (tylko percepcja)",
        "NIE (ogolny LLM)",
        "NIE (ogolny LLM)",
    ],
    "Zastosowanie robotyczne": [
        "End-to-end sterowanie",
        "Deploy na robocie",
        "Analiza sceny, planowanie",
        "Planowanie, reasoning",
        "Lekki planer, szybki",
    ],
}

df = pd.DataFrame(data)
print("=== Porownanie modeli Gemini dla robotyki ===")
print()
print(df.to_string(index=False))
print()
print("Uwaga: W tym notebooku uzywamy Gemini 2.0 Flash jako zastepstwo")
print("za Gemini Robotics-ER (kompatybilne API, podobne mozliwosci percepcji).")

## Wnioski i ograniczenia

### Co potrafi Gemini-ER w robotyce?
- Rozumienie sceny z obrazu kamery robota
- Generowanie strukturalnych planow (JSON) dla pick-and-place
- Re-planning — ponowna analiza gdy scena sie zmieni
- Rozumienie kontekstu przestrzennego ("kubek na stole", "po lewej stronie")

### Czego Gemini-ER NIE potrafi?
- **Nie generuje akcji motorycznych** — nie zwraca pozycji stawow ani momentow
- **Nie jest VLA** — nie laczy percepcji z kontrola w jednym modelu
- **Nie dziala w czasie rzeczywistym** — latencja API (~0.5-2s) jest za duza na kontrole 10 Hz
- **Nie rozumie kinematyki** — nie wie, jak robot moze sie poruszac

### Kluczowy wniosek

Gemini Robotics-ER to **warstwa percepcji i planowania**, nie pelny system sterowania. W produkcyjnym systemie robotycznym musi wspolpracowac z:
- **VLA** (np. GR00T N1.6, UnifoLM-VLA-0) — zamiana planu na akcje motoryczne
- **WBC / kontroler niskiego poziomu** — fizyczna realizacja ruchow

Pelny VLA od Google (Gemini Robotics) nie jest jeszcze publicznie dostepny.